In [55]:
import os
import uuid
from tqdm import tqdm
from dotenv import load_dotenv

from openai import OpenAI

from pinecone import Pinecone, ServerlessSpec

from sentence_transformers import SentenceTransformer

In [56]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

PINECONE_INDEX = os.getenv("PINECONE_INDEX")

print("Environment loaded successfully")

Environment loaded successfully


In [57]:
client = OpenAI(
    api_key=OPENAI_API_KEY
)

print("OpenAI initialized")

OpenAI initialized


In [58]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Embedding model loaded


In [59]:
pc = Pinecone(
    api_key=PINECONE_API_KEY
)

index_name = PINECONE_INDEX

existing_indexes = [
    index["name"]
    for index in pc.list_indexes()
]

if index_name not in existing_indexes:

    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

print("Pinecone initialized")

Pinecone initialized


In [60]:
from IPython.core import display_functions
DATA_FOLDER = "../data"
import os 
documents = []

for file_name in os.listdir(DATA_FOLDER):

    if file_name.endswith(".txt"):

        path = os.path.join(
            DATA_FOLDER,
            file_name
        )

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            text = f.read()

        documents.append({
            "source": file_name,
            "text": text
        })

print(f"Loaded {len(documents)} documents")

Loaded 6 documents


In [61]:
def chunk_text(
    text,
    chunk_size=200,
    overlap=50
):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        chunks.append(chunk)

        start += (chunk_size - overlap)

    return chunks

In [62]:
all_chunks = []

for doc in documents:

    chunks = chunk_text(doc["text"])

    for chunk in chunks:

        all_chunks.append({
            "id": str(uuid.uuid4()),
            "source": doc["source"],
            "chunk": chunk
        })

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 10


In [63]:
chunk_texts = [
    item["chunk"]
    for item in all_chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Embeddings generated")

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.55it/s]

Embeddings generated


In [64]:
vectors = []

for item, embedding in zip(all_chunks, embeddings):

    vectors.append(
        (
            item["id"],
            embedding.tolist(),
            {
                "source": item["source"],
                "text": item["chunk"]
            }
        )
    )

batch_size = 100

for i in tqdm(
    range(0, len(vectors), batch_size)
):

    batch = vectors[i:i+batch_size]

    index.upsert(batch)

print("Vectors stored successfully")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:02<00:00,  2.38s/it]

Vectors stored successfully


In [65]:
def search_documents(
    question,
    top_k=5
):

    query_embedding = embedding_model.encode(
        question
    ).tolist()

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    return results

In [74]:
question = "What is the Cancellation policy"

results = search_documents(question)

results

{'matches': [{'id': '13bae609-cab6-4158-aed1-8999f72f60f7',
              'metadata': {'source': 'appointment_policy.txt',
                           'text': 'depending on provider availability. '
                                   'Cancellation Policy: Patients should '
                                   'provide at least 24 hours notice for '
                                   'appointment cancellations. No-Show Policy: '
                                   'Repeated missed appointments without '
                                   'notice may result in scheduling '
                                   'restrictions. Specialist Referrals: '
                                   'Certain specialty appointments require '
                                   'referral approval from a primary care '
                                   'provider. Telehealth Appointments: '
                                   'Eligible appointments may be converted to '
                                   'telehealth 

In [76]:
def retrieve_context(
    question,
    top_k=3,
    threshold=0.40
):

    results = search_documents(
        question,
        top_k
    )

    contexts = []

    for match in results["matches"]:

        score = match["score"]

        if score >= threshold:

            contexts.append({
                "score": score,
                "source": match["metadata"]["source"],
                "text": match["metadata"]["text"]
            })

    return contexts

In [77]:
contexts = retrieve_context(
    "What is the Cancellation policy"
)

contexts

[{'score': 0.492134124,
  'source': 'appointment_policy.txt',
  'text': 'depending on provider availability. Cancellation Policy: Patients should provide at least 24 hours notice for appointment cancellations. No-Show Policy: Repeated missed appointments without notice may result in scheduling restrictions. Specialist Referrals: Certain specialty appointments require referral approval from a primary care provider. Telehealth Appointments: Eligible appointments may be converted to telehealth sessions based on provider approval and patient condition. Emergency Visits: Patients with emergency medical conditions should seek immediate emergency care rather than scheduling routine appointments.'},
 {'score': 0.47426036,
  'source': 'appointment_policy.txt',
  'text': 'APPOINTMENT SCHEDULING POLICY Purpose: This policy explains procedures for scheduling, rescheduling, and canceling patient appointments. Scheduling Methods: Appointments may be scheduled through: - Online patient portal - Telep

In [78]:
def build_prompt(
    question,
    contexts
):

    context_text = "\n\n".join([

        f"""
SOURCE: {c['source']}

CONTENT:
{c['text']}
"""

        for c in contexts
    ])

    prompt = f"""
You are a professional healthcare AI assistant.

IMPORTANT RULES:
- Answer ONLY from provided context
- Do NOT hallucinate
- Do NOT guess
- Keep answers short and professional
- If answer is unavailable reply:
"Sorry, I don't know based on the provided documents."

CONTEXT:
{context_text}

QUESTION:
{question}

ANSWER:
"""

    return prompt

In [79]:
def generate_answer(question):

    contexts = retrieve_context(question)

    if len(contexts) == 0:

        return {
            "answer": "Sorry, I don't know based on the provided documents.",
            "sources": []
        }

    prompt = build_prompt(
        question,
        contexts
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.2
    )

    answer = response.choices[0].message.content

    return {
        "answer": answer,
        "sources": contexts
    }

In [80]:
response = generate_answer(
    "What is the Cancellation policy"
)

response

{'answer': 'Patients should provide at least 24 hours notice for appointment cancellations.',
 'sources': [{'score': 0.492134124,
   'source': 'appointment_policy.txt',
   'text': 'depending on provider availability. Cancellation Policy: Patients should provide at least 24 hours notice for appointment cancellations. No-Show Policy: Repeated missed appointments without notice may result in scheduling restrictions. Specialist Referrals: Certain specialty appointments require referral approval from a primary care provider. Telehealth Appointments: Eligible appointments may be converted to telehealth sessions based on provider approval and patient condition. Emergency Visits: Patients with emergency medical conditions should seek immediate emergency care rather than scheduling routine appointments.'},
  {'score': 0.47426036,
   'source': 'appointment_policy.txt',
   'text': 'APPOINTMENT SCHEDULING POLICY Purpose: This policy explains procedures for scheduling, rescheduling, and canceling p

In [81]:
def ask(question):

    print(f"\nQUESTION:\n{question}")

    result = generate_answer(question)

    print("\nANSWER:\n")
    print(result["answer"])

    print("\nSOURCES:\n")

    for s in result["sources"]:

        print("-" * 50)

        print(f"Source: {s['source']}")

        print(f"Score: {round(s['score'], 3)}")

        print(f"Preview: {s['text'][:200]}")

In [83]:
ask("What is the Cancellation policy")


QUESTION:
What is the Cancellation policy

ANSWER:

Patients should provide at least 24 hours notice for appointment cancellations.

SOURCES:

--------------------------------------------------
Source: appointment_policy.txt
Score: 0.492
Preview: depending on provider availability. Cancellation Policy: Patients should provide at least 24 hours notice for appointment cancellations. No-Show Policy: Repeated missed appointments without notice may
--------------------------------------------------
Source: appointment_policy.txt
Score: 0.474
Preview: APPOINTMENT SCHEDULING POLICY Purpose: This policy explains procedures for scheduling, rescheduling, and canceling patient appointments. Scheduling Methods: Appointments may be scheduled through: - On
--------------------------------------------------
Source: appointment_policy.txt
Score: 0.404
Preview: APPOINTMENT SCHEDULING POLICY Purpose: This policy explains procedures for scheduling, rescheduling, and canceling patient appointments. Sched